In [ ]:
# -*- coding: utf-8 -*-
"""Exam_LLM_GPT01.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/16ch9_nMlDjw9QFuofKwFN1OZGEGTBKan
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 간단한 예시 문장
sentences = [
    # 자연어 생성 문장 데이터
    "나는 어제 친구와 영화를 봤다",
    "오늘 날씨가 맑고 기분이 좋다",
    "GPT는 다음 단어를 예측하며 학습한다",
    "수업 시작 합시다",
    "수업 시작시 준비할 내용입니다.",

    # [요약] 태스크 예시 (접두어: "요약: ")
    # ※ 수정: 입력과 정답을 별도 문장으로 두지 않고, "->" 로 한 시퀀스에 이어붙임
    #   (GPT는 한 시퀀스 안에서만 다음 단어를 학습하므로, 따로 두면 입력→정답 관계를 절대 배우지 못함)
    # ※ 추가 수정: 정답 끝에 종료 토큰 <EOS> 추가 → 모델이 "여기서 답이 끝났다"를 학습
    "요약: 오늘 날씨가 맑고 기분이 좋아서 공원에 갔다 나는 행복했다 -> 오늘 공원 산책 행복 <EOS>",
    "요약: 인공지능은 인간의 지능을 모방하여 학습 추론 문제 해결 등을 수행하는 기술이다 -> 인공지능은 인간 지능 모방 기술 <EOS>",

    # [번역] 태스크 예시 (접두어: "번역: ")
    "번역: I love you very much -> 너무 사랑해 <EOS>",
    "번역: Good morning everyone -> 좋은 아침이에요 <EOS>",

    # [대화] 태스크 예시 (접두어: "대화: ")
    "대화: 안녕하세요? -> 안녕하세요! 반갑습니다 <EOS>",
    "대화: 오늘 기분은 어때? -> 기분이 아주 좋아요 <EOS>",
]

# 단어 → 인덱스 딕셔너리 만들기
words = list(set(" ".join(sentences).split()))
print(words)
vocab = {w: i+2 for i, w in enumerate(words)}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1
vocab_size = len(vocab)
print(vocab)

vocab_size

# GPT용 데이터셋: input → target (한 칸 shift)
# 예: [나는, 어제, 친구와] → [어제, 친구와, 영화를]
# ※ 수정: "->" 가 있는 문장(요약/번역/대화 태스크)은 "->" 이전(prompt) 부분을
#         예측하는 위치의 정답을 -100으로 마스킹해서 loss 계산에서 제외함.
#         즉 모델은 "prompt를 잘 외우는 것"이 아니라 "prompt 다음에 올 답"만 학습하게 됨.
#         "->" 가 없는 일반 문장은 그대로 전체를 학습(평범한 다음 단어 예측).
class NextWordDataset(Dataset):
    def __init__(self, sentences, vocab):
        self.data = []
        arrow_id = vocab.get("->")
        for s in sentences:
            tokens = [vocab.get(w, 1) for w in s.split()]
            split_idx = tokens.index(arrow_id) if arrow_id in tokens else -1
            self.data.append((tokens, split_idx))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        tokens, split_idx = self.data[idx]
        x = torch.tensor(tokens[:-1])  # 입력: 마지막 단어 제외

        y = []
        for i, t in enumerate(tokens[1:]):       # y[i] = tokens[i+1] 예측
            if split_idx != -1 and (i + 1) <= split_idx:
                y.append(-100)                    # prompt(+"->") 영역 → loss 제외
            else:
                y.append(t)                       # 정답(answer) 영역 → loss 계산
        y = torch.tensor(y)

        return x, y

"""# 미니 GPT 모델 정의"""

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()

        # 1. 단어를 벡터로 변환 (Embedding)
        self.embed = nn.Embedding(vocab_size, d_model)

        # 2. 위치 정보 추가 (Positional Encoding)
        self.pos_embed = nn.Embedding(512, d_model)

        # 3. Transformer Decoder (Causal Self-Attention)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # 4. 최종 출력: 각 단어의 확률 분포
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        pos = torch.arange(seq_len).unsqueeze(0)

        # 단어 + 위치 임베딩
        h = self.embed(x) + self.pos_embed(pos)

        # Causal Mask: 미래 단어 가리기 🔒
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len)

        # Transformer 통과 (Self-Attention → FFN)
        out = self.transformer(h, h, tgt_mask=mask)

        # vocab_size 차원으로 변환 → 각 단어 확률
        return self.fc_out(out)

"""# ③ 학습 루프 (Training Loop)"""

# 모델 / 손실함수 / 옵티마이저 초기화
model     = MiniGPT(vocab_size)
criterion = nn.CrossEntropyLoss(ignore_index=-100)  # prompt 영역(마스킹된 부분) 무시
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loader    = DataLoader(NextWordDataset(sentences, vocab), batch_size=1)

for epoch in range(700):
    total_loss = 0
    for x, y in loader:
        # 1. Forward: 다음 단어 예측
        logits = model(x)                            # (B, T, vocab)
        loss   = criterion(logits.view(-1, vocab_size),
                           y.view(-1))              # 정답과 비교

        # 2. Backward: 오차 역전파
        optimizer.zero_grad()
        loss.backward()

        # 3. Step: 파라미터 업데이트
        optimizer.step()
        total_loss += loss.item()

    if (epoch+1) % 100 == 0:
        print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")

"""# 텍스트 생성 (Inference)"""

# 텍스트 생성(Inference)
def generate(model, prompt, vocab, max_new_tokens=10):
    idx2word = {v: k for k, v in vocab.items()}
    tokens   = [vocab.get(w, 1) for w in prompt.split()]
    eos_id   = vocab.get("<EOS>")   # ※ 추가: 종료 토큰 id
    model.eval()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            x      = torch.tensor(tokens).unsqueeze(0)
            logits = model(x)                   # 다음 단어 확률
            next_token = logits[0, -1].argmax()  # 가장 높은 확률 선택
            tokens.append(next_token.item())

            # ※ 추가: <EOS>가 나오면 더 생성하지 않고 즉시 멈춤
            if eos_id is not None and next_token.item() == eos_id:
                break

    # 화면에 보여줄 때는 <EOS> 토큰 자체는 숨김
    return " ".join(idx2word.get(t, "?") for t in tokens if t != eos_id)

# 예시 함수 추가
# ※ 수정 1: translate/chat이 잘못 복붙된 "요약:" 대신 각자 맞는 prefix("번역:", "대화:")를 쓰도록 수정
# ※ 수정 2: 학습데이터를 "입력 -> 정답" 형태로 합쳤으므로, 추론 시에도 prompt 끝에 "->"를 붙여야
#          모델이 "이제부터 정답을 이어 써라"라는 신호를 받음
# ※ 수정 3: max_new_tokens은 "최대 한도"일 뿐, 실제로는 <EOS>가 나오면 그 전에 멈춤
def summarize(model, prompt, vocab):
    prompt = f"요약: {prompt} ->"
    return generate(model, prompt, vocab, max_new_tokens=10)

def translate(model, prompt, vocab):
     prompt = f"번역: {prompt} ->"
     return generate(model, prompt, vocab, max_new_tokens=10)

def chat(model, prompt, vocab):
     prompt = f"대화: {prompt} ->"
     return generate(model, prompt, vocab, max_new_tokens=10)

# 사용 예시
result = generate(model, "수업 시작", vocab)
print(result)  # 나는 어제 친구와 영화를 봤다

#  테스트할 입력 데이터
test_text_sum = "오늘 날씨가 맑고 기분이 좋아서 공원에 갔다 나는 행복했다"
test_text_trans = "I love you very much"
test_text_chat = "안녕하세요?"

print(f"\n📝 [문장 요약] 입력: '{test_text_sum}'")
print(f"   ➔ 결과: {summarize(model, test_text_sum, vocab)}")

print(f"\n [문장 번역] 입력: '{test_text_trans}'")
print(f"   ➔ 결과: {translate(model, test_text_trans, vocab)}")

print(f"\n💬 [간이 대화] 입력: '{test_text_chat}'")
print(f"   ➔ 결과: {chat(model, test_text_chat, vocab)}")

['공원', '맑고', '기분이', '좋아요', 'GPT는', '내용입니다.', '너무', '행복했다', '산책', '인간의', '시작시', '등을', 'I', '봤다', 'you', '영화를', '학습', '행복', '어때?', '모방하여', '요약:', '나는', '어제', '번역:', '예측하며', '수업', '지능을', '반갑습니다', '모방', '<EOS>', '친구와', '사랑해', '단어를', '해결', '수행하는', '기술이다', '문제', 'much', '학습한다', '지능', 'everyone', '아침이에요', '오늘', '갔다', 'love', '기분은', 'Good', '대화:', '날씨가', 'very', 'morning', '아주', '준비할', '인공지능은', '좋아서', '안녕하세요!', '기술', '안녕하세요?', '좋은', '시작', '합시다', '->', '공원에', '인간', '추론', '다음', '좋다']
{'공원': 2, '맑고': 3, '기분이': 4, '좋아요': 5, 'GPT는': 6, '내용입니다.': 7, '너무': 8, '행복했다': 9, '산책': 10, '인간의': 11, '시작시': 12, '등을': 13, 'I': 14, '봤다': 15, 'you': 16, '영화를': 17, '학습': 18, '행복': 19, '어때?': 20, '모방하여': 21, '요약:': 22, '나는': 23, '어제': 24, '번역:': 25, '예측하며': 26, '수업': 27, '지능을': 28, '반갑습니다': 29, '모방': 30, '<EOS>': 31, '친구와': 32, '사랑해': 33, '단어를': 34, '해결': 35, '수행하는': 36, '기술이다': 37, '문제': 38, 'much': 39, '학습한다': 40, '지능': 41, 'everyone': 42, '아침이에요': 43, '오늘': 44, '갔다': 45, 'love': 46, '기분은': 47, 'Good': 48, '대화:':